Neste notebook estarei aprendendo a aplicar técnicas de aumento de imagem em datasets para treinar redes neurais.

O aumento de imagem gera novas imagens para o dataset aplicando transformações nela, como girar a imagem, alterar o zoom, etc...

Isso é especialmente útil quando se deseja treinar uma rede neural com um dataset não muito grande, pois dessa forma o dataset "aumenta" de tamanho.

Importação das bibliotecas:

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
from torchvision import datasets
import torchvision.transforms as transforms

Definção da base de dados:

In [2]:
torch.manual_seed(123)

Cria um tranformador composto por três transformações:

rhf: Gira aleatoriamente as imagens na horizontal
ra: Aplica aleatóriamente  nas imagens as seguintes transformaçoes afim:

    - Rotaciona a imagem entre -7 e 7 graus
    - translada aleatoriamente a imagem em:
     0% no eixo horizontal e de 0% a 7% no eixo vertical
    - Inclina a imagem entre -7 e 7 graus
        Inclinar significa girar a parte de cima da imagem sem girar a base, ou vice versa.
    - Da zoom na imagem entre 1x e 1.2x (Zoom e 0% a 20%)

transformer: Transforma a imagem em tensor e aplica a normalização de escala

In [ ]:

rhf = transforms.RandomHorizontalFlip()
ra = transforms.RandomAffine(degrees = 7, 
                             translate = (0, 0.07), # horizontal and vertical shifts
                             shear = 7,
                             scale = (1, 1.2)) # zoom range
transformer = transforms.ToTensor()

transform_train = transforms.Compose([rhf, ra, transformer])

In [4]:
transform_test = transforms.ToTensor()

Define os datasets de treino e teste e os loaders para cada um

In [ ]:
train = datasets.MNIST(root = '.', train = True, download = True, transform = transform_test)
test = datasets.MNIST(root = '.', train = False, download = True, transform = transform_test)

In [ ]:
# Usarei novamente os loaders do pytorch
train_loader = torch.utils.data.DataLoader(train, batch_size = 1024)
test_loader = torch.utils.data.DataLoader(test, batch_size = 1024)

Definição da rede neural:

In [7]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

In [8]:
class classificador_torch(nn.Module):
  def __init__(self):
    super().__init__()

    self.conv1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=(3,3))
    self.conv2 = nn.Conv2d(32, 32, (3, 3))
    self.activation = nn.ReLU()
    self.bnorm = nn.BatchNorm2d(num_features=32)
    self.pool = nn.MaxPool2d(kernel_size = (2,2))
    self.flatten = nn.Flatten()

    self.linear1 = nn.Linear(in_features=32*5*5, out_features=128)
    self.linear2 = nn.Linear(128, 128)
    self.output = nn.Linear(128, 10)
    self.dropout = nn.Dropout(p = 0.2)

    self.model = nn.Sequential(
      self.conv1,
      self.activation,
      self.bnorm,
      self.pool,

      self.conv2,
      self.activation,
      self.bnorm,
      self.pool,

      self.flatten,

      self.linear1,
      self.activation,
      self.dropout,

      self.linear2,
      self.activation,
      self.dropout,

      self.output
    ).to(device)

  def forward(self, X):
    return self.model(X)

Cria-se os objetos da rede, função custo e otimizador

In [9]:
net = classificador_torch()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters())

net.to(device)

classificador_torch(
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1))
  (conv2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1))
  (activation): ReLU()
  (bnorm): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool): MaxPool2d(kernel_size=(2, 2), stride=(2, 2), padding=0, dilation=1, ceil_mode=False)
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear1): Linear(in_features=800, out_features=128, bias=True)
  (linear2): Linear(in_features=128, out_features=128, bias=True)
  (output): Linear(in_features=128, out_features=10, bias=True)
  (dropout): Dropout(p=0.2, inplace=False)
  (model): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1))
    (1): ReLU()
    (2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): MaxPool2d(kernel_size=(2, 2), stride=(2, 2), padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1))
    (5): ReLU()
    (6): 

Define-se loop de treinamento:

In [10]:
def training_loop(loader, epoch, training = True):
    epoch_loss = 0
    epoch_accuracy = 0

    for batch_number, data in enumerate(loader):
        inputs, labels = data

        inputs, labels = inputs.to(device), labels.to(device)

        if (training):
            optimizer.zero_grad()

        # Faz o forward
        outputs = net(inputs)

        loss = criterion(outputs, labels)

        if (training):
            # Calcula os pesos e biases com a descida do gradiente
            loss.backward()

            # Aplica os pesos e biases calculados acima
            optimizer.step()

        epoch_loss = epoch_loss + loss.item()

        # Aplica softmax no output da rede para obter uma matriz de probabilidades, cada linha é um array de probabilidades associados a uma das imagens do batch.
        probs = F.softmax(outputs, dim = 1)

        # Para cada output do batch, pega os k maiores valores no array de probabilidades acima (1 valor), ao longo do eixo das linhas (classes),
        # e o indice desse valor, que representa a classe. Cada neuronio representa também um indice, por isso os indices entre top_class e top_labels se correspondem.
        top_prob, top_class = probs.topk(k = 1, dim = 1)

        # Cria um tensor de valores booleanos, cada valor representa o erro ou acerto da rede neural em relação ao input.
        # labels.view ajusta o tensor automaticamente para ter um shape igual ao seu argumento
        # *top_class.shape passa cada dimensao do shape de top_class como argumento para o remodelador.
        equals = top_class == labels.view(*top_class.shape)

        # equals.type(torch.float) converte cada valor em equals para 1.0 ou 0.0
        # torch.mean tira a media desses valores todos
        accuracy = torch.mean(equals.type(torch.float))

        epoch_accuracy = epoch_accuracy + accuracy

        print('\rÉpoca {:3d} - Loop {:3d} de {:3d}: perda {:03.2f} - precisão {:03.2f}'.format(epoch + 1, batch_number + 1, len(loader), loss, 
                                   accuracy), end = '\r')
        
    print('\rÉPOCA {:3d} FINALIZADA: perda {:.5f} - precisão {:.5f}'.format(epoch+1, epoch_loss/len(loader), 
                     epoch_accuracy/len(loader)))
    

Treina o modelo por 10 epochs:

In [11]:
for epoch in range(10):
    # Treina a rede em uma epoch
    print("Treino...")
    training_loop(train_loader, epoch)

    # Coloca a rede em modo de avaliação (desabilita a camada de Dropout, e faz a camada de normalização de batch, usar media total e variancia total.)
    net.eval()

    print("Teste...")
    training_loop(test_loader, epoch, training=False)

Treino...
ÉPOCA   1 FINALIZADA: perda 0.88410 - precisão 0.707960
Teste...
ÉPOCA   1 FINALIZADA: perda 2.10843 - precisão 0.369538
Treino...
ÉPOCA   2 FINALIZADA: perda 0.47644 - precisão 0.849255
Teste...
ÉPOCA   2 FINALIZADA: perda 0.16152 - precisão 0.952905
Treino...
ÉPOCA   3 FINALIZADA: perda 0.21837 - precisão 0.932535
Teste...
ÉPOCA   3 FINALIZADA: perda 0.10858 - precisão 0.966426
Treino...
ÉPOCA   4 FINALIZADA: perda 0.15979 - precisão 0.950966
Teste...
ÉPOCA   4 FINALIZADA: perda 0.10129 - precisão 0.966526
Treino...
ÉPOCA   5 FINALIZADA: perda 0.13193 - precisão 0.959427
Teste...
ÉPOCA   5 FINALIZADA: perda 0.09062 - precisão 0.970087
Treino...
ÉPOCA   6 FINALIZADA: perda 0.10821 - precisão 0.966797
Teste...
ÉPOCA   6 FINALIZADA: perda 0.07201 - precisão 0.976017
Treino...
ÉPOCA   7 FINALIZADA: perda 0.09699 - precisão 0.969978
Teste...
ÉPOCA   7 FINALIZADA: perda 0.07184 - precisão 0.976817
Treino...
ÉPOCA   8 FINALIZADA: perda 0.08822 - precisão 0.971938
Teste...
ÉPOCA   